# 第 8 周评估：智能体交易定价器（Agentic Deal Pricer）

## 练习目标

搭一条小型 **Agent（智能体）** 管道，把检索与工具调用串起来：

1. 把少量交易（deals）嵌入 **Chroma** 向量库
2. 用 **RAG（检索增强生成）** 查出相关交易
3. 用 LLM **总结**交易、**估算**公允价
4. 让规划器（planner）产出工具调用计划，再由执行器（executor）逐步跑完

## 和本课第 8 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 向量检索 / RAG | `SentenceTransformer` + Chroma `query` |
| Tool calling 规划 | `plan()` 让模型返回 JSON 工具列表 |
| 多步执行 | `execute_plan()` 按工具名分发并维护 context |

## 怎么跑

1. `.env` 里准备 `OPENAI_API_KEY`；可选 `OPENAI_BASE_URL`、`OPENAI_MODEL`（默认走 OpenRouter）
2. 依次运行：建索引 → 定义工具 → 规划 → 执行
3. 最后一格会真实调用 LLM，需有效 API Key


In [1]:
# ========== 导入：配置、向量库、嵌入、LLM 客户端 ==========

# json：解析 planner 返回的工具调用计划
import json
# os：读环境变量里的 API Key / Base URL / Model
import os
# dataclass：本格导入（后续可扩展结构化配置；逻辑保持原样）
from dataclasses import dataclass

# Chroma：轻量向量数据库，存 deals 的 embedding
import chromadb
# SentenceTransformer：本地句向量模型，把交易文本编码成向量
from sentence_transformers import SentenceTransformer
# OpenAI 兼容客户端：可指向 OpenRouter 等网关
from openai import OpenAI
# load_dotenv：从 .env 加载密钥，避免写进代码
from dotenv import load_dotenv


In [2]:
# ========== 环境与 OpenAI 兼容客户端 ==========

# override=True：.env 覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 主密钥：没有则后面 LLM 调用会失败
API_KEY = os.getenv("OPENAI_API_KEY")
# 默认走 OpenRouter 的 OpenAI 兼容端点（URL 保持原样）
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
# 默认小模型；可用 OPENAI_MODEL 覆盖
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

# 缺密钥时打印英文提示（影响行为的文案不翻译）
if not API_KEY:
    print("Missing OPENAI_API_KEY in .env")
else:
    print("API key loaded")

# 创建客户端：密钥 + 自定义 base_url
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


API key loaded


In [3]:
# ========== 迷你交易数据集 + 文本化函数 ==========

# DEALS：内存里的「商品货架」；字段 id/title/category/description/ask 供检索与估价
DEALS = [
    {
        "id": "d1",
        "title": "Dell Latitude 7420 i7 Laptop",
        "category": "Electronics",
        "description": "14-inch business laptop, Intel i7, 16GB RAM, 512GB SSD, 2-year warranty, very good condition",
        "ask": 750,
    },
    {
        "id": "d2",
        "title": "Herman Miller Aeron Chair",
        "category": "Office",
        "description": "Ergonomic office chair, fully adjustable, size B, minor scuffs, no tears, mesh intact",
        "ask": 480,
    },
    {
        "id": "d3",
        "title": "Sony A6400 Camera + 16-50mm Lens",
        "category": "Photography",
        "description": "Mirrorless camera, 24MP, 4K video, includes kit lens and two batteries, shutter count low",
        "ask": 680,
    },
    {
        "id": "d4",
        "title": "Ikea Bekant Standing Desk",
        "category": "Office",
        "description": "Electric standing desk, 160x80 cm, white top, black frame, smooth height adjustment",
        "ask": 250,
    },
    {
        "id": "d5",
        "title": "Apple Watch Series 7",
        "category": "Wearables",
        "description": "45mm GPS model, good battery health, includes charger and extra sport band",
        "ask": 220,
    },
]

def deal_text(d):
    # 把结构化字段拼成一段可嵌入、可给 LLM 读的纯文本（字段名保持英文）
    return f"Title: {d['title']}\nCategory: {d['category']}\nDescription: {d['description']}"


In [4]:
# ========== 建索引：句向量 → Chroma collection ==========

# 轻量英文句向量模型；字符串 id 必须原样，才能正确下载权重
embedder = SentenceTransformer("all-MiniLM-L6-v2")
# 每条 deal 转成统一文本，再批量编码
texts = [deal_text(d) for d in DEALS]
# normalize_embeddings=True：向量单位化，适合余弦相似度检索
embeddings = embedder.encode(texts, normalize_embeddings=True)

# 进程内 Chroma 客户端（演示用，不强制持久化路径）
chroma = chromadb.Client()
# 集合名 deals：没有就创建，有则复用
collection = chroma.get_or_create_collection("deals")
# upsert：按 id 写入/更新文档、向量、元数据（含 ask 售价）
collection.upsert(
    ids=[d["id"] for d in DEALS],
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"title": d["title"], "category": d["category"], "ask": d["ask"]} for d in DEALS],
)
print("Indexed deals:", len(DEALS))


Indexed deals: 5


In [5]:
# ========== 工具 1：向量检索 search_deals ==========

def search_deals(query, k=3):
    # 查询句编码为向量（与建库时同一 embedder、同一归一化）
    q_emb = embedder.encode([query], normalize_embeddings=True)
    # 在 Chroma 里取 top-k；同时要 documents 与 metadatas
    results = collection.query(query_embeddings=q_emb, n_results=k, include=["documents", "metadatas"])
    hits = []
    # Chroma 返回按 query 分组的列表；这里只有 1 个 query，取 [0]
    for doc, meta, did in zip(results["documents"][0], results["metadatas"][0], results["ids"][0]):
        hits.append({"id": did, "doc": doc, "meta": meta})
    return hits


In [6]:
# ========== 工具 2–4：总结、估价、通知（后两者走 LLM） ==========

def llm_summarize(text):
    # 让模型把交易压成 2–3 条要点；system prompt 保持英文原样
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Summarize the deal in 2-3 bullet points."},
            {"role": "user", "content": text},
        ],
        max_tokens=150,
    )
    return resp.choices[0].message.content.strip()

def llm_estimate_price(text):
    # 要求只回一个美元数字，方便后续当价格用；prompt 不翻译
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Estimate a fair price in USD. Respond with a single number."},
            {"role": "user", "content": text},
        ],
        max_tokens=20,
    )
    return resp.choices[0].message.content.strip()

def send_notification(deal_id, price):
    # 演示用「假通知」：不发邮件，只返回排队文案
    return f"Notification queued for {deal_id} at price ${price}."


In [7]:
# ========== 规划器：让 LLM 产出 JSON 工具调用列表 ==========

# 系统提示：你是规划智能体，只返回工具调用 JSON（英文保持原样）
PLANNER_SYSTEM = "You are a planning agent. Return a JSON list of tool calls."
# Schema 示例：告诉模型有哪些 tool、参数长什么样
PLANNER_SCHEMA = {
    "tools": [
        {"tool": "search_deals", "args": {"query": "...", "k": 3}},
        {"tool": "llm_summarize", "args": {"text": "..."}},
        {"tool": "llm_estimate_price", "args": {"text": "..."}},
        {"tool": "send_notification", "args": {"deal_id": "d1", "price": "..."}},
    ]
}

def plan(query):
    # 用户侧说明：先检索，再总结+估价，且只回 JSON（prompt 字符串不翻译）
    prompt = (
        "Create a short plan of tool calls to answer the user. "
        "Use search_deals first, then summarize and estimate price. "
        "Return JSON only.\n\n"
        f"Schema example: {json.dumps(PLANNER_SCHEMA)}\n\n"
        f"User query: {query}"
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        max_tokens=300,
    )
    text = resp.choices[0].message.content.strip()
    try:
        # 理想情况：模型直接给出合法 JSON
        return json.loads(text)
    except json.JSONDecodeError:
        # 解析失败时的兜底计划：至少先检索
        return {"tools": [{"tool": "search_deals", "args": {"query": query, "k": 3}}]}


In [8]:
# ========== 执行器：按计划逐步调用工具，并维护上下文 ==========

# 工具名 → 可调用对象；planner 产出的名字必须落在这个字典里
TOOLS = {
    "search_deals": search_deals,
    "llm_summarize": llm_summarize,
    "llm_estimate_price": llm_estimate_price,
    "send_notification": send_notification,
}

def execute_plan(plan_json):
    outputs = []
    # context：步骤间共享检索命中、中间 LLM 结果
    context = {}
    for step in plan_json.get("tools", []):
        tool = step.get("tool")
        args = step.get("args", {})
        # 未知工具：记错误，继续下一步（容错）
        if tool not in TOOLS:
            outputs.append({"tool": tool, "error": "unknown tool"})
            continue

        if tool == "search_deals":
            # 检索结果写入 context["hits"]，供后续 summarize/estimate 复用
            hits = TOOLS[tool](**args)
            context["hits"] = hits
            outputs.append({"tool": tool, "result": hits})
        elif tool in ("llm_summarize", "llm_estimate_price"):
            # 若 text 缺失、为空或仍是占位符，则用第一条检索文档顶上
            if ("text" not in args or not args.get("text") or args["text"] == "<search_deals_results>") and context.get("hits"):
                args = {"text": context["hits"][0]["doc"]}
            result = TOOLS[tool](**args)
            context[tool] = result
            outputs.append({"tool": tool, "result": result})
        else:
            # 例如 send_notification：直接按 args 调用
            result = TOOLS[tool](**args)
            outputs.append({"tool": tool, "result": result})
    return outputs


In [10]:
# ========== 端到端演示：规划 → 执行 → 打印 ==========

# 用户自然语言查询（保持英文，与 deals 语料一致）
query = "Find a good deal for a used business laptop with 16GB RAM and estimate a fair price."
# 让规划器产出 JSON 计划
plan_json = plan(query)
print("PLAN:\n", json.dumps(plan_json, indent=2))
# 执行器逐步跑工具
results = execute_plan(plan_json)
print("\nRESULTS:\n", json.dumps(results, indent=2))


PLAN:
 {
  "tools": [
    {
      "tool": "search_deals",
      "args": {
        "query": "used business laptop 16GB RAM",
        "k": 3
      }
    },
    {
      "tool": "llm_summarize",
      "args": {
        "text": ""
      }
    },
    {
      "tool": "llm_estimate_price",
      "args": {
        "text": ""
      }
    }
  ]
}

RESULTS:
 [
  {
    "tool": "search_deals",
    "result": [
      {
        "id": "d1",
        "doc": "Title: Dell Latitude 7420 i7 Laptop\nCategory: Electronics\nDescription: 14-inch business laptop, Intel i7, 16GB RAM, 512GB SSD, 2-year warranty, very good condition",
        "meta": {
          "category": "Electronics",
          "ask": 750,
          "title": "Dell Latitude 7420 i7 Laptop"
        }
      },
      {
        "id": "d4",
        "doc": "Title: Ikea Bekant Standing Desk\nCategory: Office\nDescription: Electric standing desk, 160x80 cm, white top, black frame, smooth height adjustment",
        "meta": {
          "category": "Office"